In [5]:
import os, time, gdown

PROJ = "/home/jovyan/vault/heidelberg_lulc_2025"
missing = {
    "DW_2025_12_10m_9class.tif": "1gkYyZeC3nJZiruZdVuY-LqutcISCMlfs",
    "DW_2025_12_40m_9class.tif": "1N61loFvyRsJLrLHIPM-p1YWTgu79_53c",
}

for fname, fid in missing.items():
    out = f"{PROJ}/dynamic_world/raw/{fname}"
    if os.path.exists(out) and os.path.getsize(out) > 0:
        print("already present:", fname)
        continue
    for attempt in range(1, 4):
        try:
            gdown.download(id=fid, output=out, quiet=False)
            if os.path.exists(out) and os.path.getsize(out) > 0:
                print("OK:", fname, round(os.path.getsize(out) / 1e6, 2), "MB")
                break
        except Exception as e:
            print(f"attempt {attempt} failed for {fname}: {e}")
        time.sleep(30)
    else:
        print("STILL MISSING:", fname)

attempt 1 failed for DW_2025_12_10m_9class.tif: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1gkYyZeC3nJZiruZdVuY-LqutcISCMlfs

but Gdown can't. Please check connections and permissions.
attempt 2 failed for DW_2025_12_10m_9class.tif: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1gkYyZeC3nJZiruZdVuY-LqutcISCMlfs

but Gdown can't. Please check connections and permissions.
attempt 3 failed for DW_2025_12_10m_9class.tif: 

In [10]:
import os, glob, hashlib
import pandas as pd

PROJ = "/home/jovyan/vault/heidelberg_lulc_2025"

def md5(path, chunk=8 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

rows = []
for folder in [f"{PROJ}/sentinel1/raw", f"{PROJ}/dynamic_world/raw"]:
    for p in sorted(glob.glob(f"{folder}/**/*", recursive=True)):
        if os.path.isfile(p):
            rows.append({"folder": os.path.relpath(os.path.dirname(p), PROJ),
                         "file": os.path.basename(p),
                         "size_MB": round(os.path.getsize(p) / 1e6, 2),
                         "md5": md5(p)})

manifest = pd.DataFrame(rows)
manifest.to_csv(f"{PROJ}/logs/raw_file_manifest.csv", index=False)

pd.set_option("display.max_rows", 200)
print(manifest.groupby("folder").agg(files=("file", "count"), total_MB=("size_MB", "sum")))
print()
print(manifest[["folder", "file", "size_MB"]].to_string(index=False))

                   files  total_MB
folder                            
dynamic_world/raw     24    947.97
sentinel1/raw         24   2103.42

           folder                        file  size_MB
    sentinel1/raw S1_2025_Apr_Orbit15_10m.tif   155.80
    sentinel1/raw S1_2025_Apr_Orbit15_40m.tif    20.44
    sentinel1/raw S1_2025_Aug_Orbit15_10m.tif   154.71
    sentinel1/raw S1_2025_Aug_Orbit15_40m.tif    20.41
    sentinel1/raw S1_2025_Dec_Orbit15_10m.tif   154.52
    sentinel1/raw S1_2025_Dec_Orbit15_40m.tif    20.38
    sentinel1/raw S1_2025_Feb_Orbit15_10m.tif   155.49
    sentinel1/raw S1_2025_Feb_Orbit15_40m.tif    20.42
    sentinel1/raw S1_2025_Jan_Orbit15_10m.tif   155.36
    sentinel1/raw S1_2025_Jan_Orbit15_40m.tif    20.40
    sentinel1/raw S1_2025_Jul_Orbit15_10m.tif   154.61
    sentinel1/raw S1_2025_Jul_Orbit15_40m.tif    20.39
    sentinel1/raw S1_2025_Jun_Orbit15_10m.tif   154.94
    sentinel1/raw S1_2025_Jun_Orbit15_40m.tif    20.40
    sentinel1/raw S1_2025_Mar_Orbi

In [7]:
import os, hashlib

RAW = "/home/jovyan/vault/heidelberg_lulc_2025/dynamic_world/raw"
orig = f"{RAW}/DW_2025_12_10m_9class.tif"
copy = f"{RAW}/DW_2025_12_10m_9class-Copy1.tif"

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()

def info(p):
    return (os.path.getsize(p) / 1e6, md5(p)) if os.path.exists(p) else (None, None)

o_size, o_md5 = info(orig)
c_size, c_md5 = info(copy)
print(f"original: {o_size} MB  md5={o_md5}")
print(f"copy    : {c_size} MB  md5={c_md5}")

if c_size is None:
    print("No copy found - nothing to do.")
elif o_md5 == c_md5:
    os.remove(copy)
    print("Identical -> deleted the copy.")
elif o_size is None or o_size < c_size:
    os.replace(copy, orig)
    print("Original was missing/smaller (incomplete).")
else:
    print("Files differ and the original is not smaller. Nothing changed.")

print("\nDecember files now present:")
for f in sorted(os.listdir(RAW)):
    if "_12_" in f:
        print(f"  {f:35s} {os.path.getsize(os.path.join(RAW, f))/1e6:8.2f} MB")

original: 88.007005 MB  md5=598b7b5017f8ebfc7b898d530d318d0b
copy    : 55.574528 MB  md5=444bd56a4553ed4eaf8ed766927bb44c
Files differ and the original is not smaller. Nothing changed - send me this output.

December files now present:
  DW_2025_12_10m_9class-Copy1.tif        55.57 MB
  DW_2025_12_10m_9class.tif              88.01 MB
  DW_2025_12_40m_9class.tif               6.23 MB


In [9]:
import site
site.addsitedir("/home/jovyan/vault/heidelberg_lulc_2025/pylibs/lib/python3.11/site-packages")
import os, glob, rasterio

RAW = "/home/jovyan/vault/heidelberg_lulc_2025/dynamic_world/raw"
orig = f"{RAW}/DW_2025_12_10m_9class.tif"
copy = f"{RAW}/DW_2025_12_10m_9class-Copy1.tif"

print("Sizes of the other 10 m DW files (for comparison):")
for p in sorted(glob.glob(f"{RAW}/DW_2025_*_10m_9class.tif")):
    print(f"  {os.path.basename(p):32s} {os.path.getsize(p)/1e6:7.2f} MB")

def fully_readable(p):
    try:
        with rasterio.open(p) as src:
            for i in range(1, src.count + 1):
                src.read(i)                      # forces every pixel to be decoded
            return True, f"{src.count} bands, {src.width}x{src.height}, {src.dtypes[0]}"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

ok_o, msg_o = fully_readable(orig)
ok_c, msg_c = fully_readable(copy)
print(f"\noriginal readable: {ok_o}  -> {msg_o}")
print(f"copy     readable: {ok_c}  -> {msg_c}")

if ok_o and not ok_c:
    os.remove(copy)
    print("\nOriginal is complete, copy is broken -> deleted the copy.")
else:
    print("\nNot deleting anything .")

Sizes of the other 10 m DW files (for comparison):
  DW_2025_01_10m_9class.tif          38.94 MB
  DW_2025_02_10m_9class.tif          86.82 MB
  DW_2025_03_10m_9class.tif          85.33 MB
  DW_2025_04_10m_9class.tif          84.49 MB
  DW_2025_05_10m_9class.tif          81.67 MB
  DW_2025_06_10m_9class.tif          81.23 MB
  DW_2025_07_10m_9class.tif          78.81 MB
  DW_2025_08_10m_9class.tif          80.51 MB
  DW_2025_09_10m_9class.tif          75.13 MB
  DW_2025_10_10m_9class.tif          70.95 MB
  DW_2025_11_10m_9class.tif          32.32 MB
  DW_2025_12_10m_9class.tif          88.01 MB

original readable: True  -> 3 bands, 4001x4001, float32
copy     readable: False  -> RasterioIOError: Read failed. See previous exception for details.

Original is complete, copy is broken -> deleted the copy.


In [11]:
import site
site.addsitedir("/home/jovyan/vault/heidelberg_lulc_2025/pylibs/lib/python3.11/site-packages")
import os, glob, hashlib, rasterio
import pandas as pd

PROJ = "/home/jovyan/vault/heidelberg_lulc_2025"

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()

def check(p):
    try:
        with rasterio.open(p) as src:
            for i in range(1, src.count + 1):
                src.read(i)
            return "OK", src.count, src.width, src.height
    except Exception as e:
        return f"BROKEN: {type(e).__name__}", None, None, None

rows = []
for folder in [f"{PROJ}/sentinel1/raw", f"{PROJ}/dynamic_world/raw"]:
    for p in sorted(glob.glob(f"{folder}/*.tif")):
        status, bands, w, h = check(p)
        rows.append({"folder": os.path.relpath(folder, PROJ), "file": os.path.basename(p),
                     "size_MB": round(os.path.getsize(p) / 1e6, 2), "readable": status,
                     "bands": bands, "width": w, "height": h, "md5": md5(p)})
        print(f"{status:10s} {os.path.basename(p)}")

manifest = pd.DataFrame(rows)
manifest.to_csv(f"{PROJ}/logs/raw_file_manifest.csv", index=False)

print("\n=== SUMMARY ===")
print(manifest.groupby("folder").agg(files=("file", "count"),
                                     total_MB=("size_MB", "sum"),
                                     broken=("readable", lambda s: (s != "OK").sum())))
print("\nOther files in the raw folders (not .tif):")
for folder in [f"{PROJ}/sentinel1/raw", f"{PROJ}/dynamic_world/raw"]:
    for p in os.listdir(folder):
        if not p.endswith(".tif"):
            print("  ", os.path.relpath(os.path.join(folder, p), PROJ))

OK         S1_2025_Apr_Orbit15_10m.tif
OK         S1_2025_Apr_Orbit15_40m.tif
OK         S1_2025_Aug_Orbit15_10m.tif
OK         S1_2025_Aug_Orbit15_40m.tif
OK         S1_2025_Dec_Orbit15_10m.tif
OK         S1_2025_Dec_Orbit15_40m.tif
OK         S1_2025_Feb_Orbit15_10m.tif
OK         S1_2025_Feb_Orbit15_40m.tif
OK         S1_2025_Jan_Orbit15_10m.tif
OK         S1_2025_Jan_Orbit15_40m.tif
OK         S1_2025_Jul_Orbit15_10m.tif
OK         S1_2025_Jul_Orbit15_40m.tif
OK         S1_2025_Jun_Orbit15_10m.tif
OK         S1_2025_Jun_Orbit15_40m.tif
OK         S1_2025_Mar_Orbit15_10m.tif
OK         S1_2025_Mar_Orbit15_40m.tif
OK         S1_2025_May_Orbit15_10m.tif
OK         S1_2025_May_Orbit15_40m.tif
OK         S1_2025_Nov_Orbit15_10m.tif
OK         S1_2025_Nov_Orbit15_40m.tif
OK         S1_2025_Oct_Orbit15_10m.tif
OK         S1_2025_Oct_Orbit15_40m.tif
OK         S1_2025_Sep_Orbit15_10m.tif
OK         S1_2025_Sep_Orbit15_40m.tif
OK         DW_2025_01_10m_9class.tif
OK         DW_2025_01_40m_9